# Spektrale Auswertungsdatei

**Tag 3, Zusatz**
**Notebook:** `tag3/spectral_eval_file.ipynb`
**Daten:** `data/data-x1.txt`

## Ziel

Du analysierst das Textformat von `data/data-x1.txt` und arbeitest mit der Klasse `SpectralEvalFile`. Die Klasse trennt Info-Block, Spaltenkopf und Messdaten.

Arbeitsverzeichnis: `python_basics-nov-2026`.


## Aufbau der Datei

Die Datei ist kein reines CSV. Sie hat drei Teile:

1. **Info-Block** zwischen `Info-Block` und `Ende Info-Block`
   - Zeitstempel der Auswertung
   - Metadaten wie Dark-Value und HDF5-Namen
   - Konfiguration (Blende, ESP, Wellenlängenraster)
   - Geometrie (Abstände, Fläche, geometrischer Faktor)
2. **Spaltenkopf** nach `***`
   - Spaltennamen: Index, WL, MPLE, DetSignal, STRAHLSTAERKE
   - Einheiten in eckigen Klammern
   - Hinweise wie `intpol`, `darkcorr`, `berechnet`
3. **Messdaten** zwischen `+++ Start of Data+++` und `---End of Data---`
   - kommagetrennte Zeilen in wissenschaftlicher Schreibweise
   - Wellenlänge von 110 nm bis 196 nm (fein bis 170 nm, danach gröberer Schritt)

Der Klassenname `SpectralEvalFile` beschreibt dieses Auswertungs-Exportformat.


## Klasse laden

Die nächste Zelle definiert `SpectralEvalFile`. Danach lädst du `data/data-x1.txt`.


In [1]:
from pathlib import Path
import re

import pandas as pd


class SpectralEvalFile:
    """Parse a spectral evaluation export (info block, header, scan data)."""

    def __init__(self, path):
        """Read the file and run the full parse."""
        self.path = Path(path)
        self.raw_text = self.path.read_text(encoding="utf-8")
        self.lines = self.raw_text.splitlines()

        self.info_title = ""
        self.timestamp = ""
        self.metadata = {}
        self.configuration = {}
        self.geometry = {}
        self.column_names = []
        self.column_units = []
        self.column_notes = []
        self.csv_text = ""
        self.data = pd.DataFrame()

        self._parse()

    @classmethod
    def from_path(cls, path):
        """Build an instance from a file path."""
        return cls(path)

    def _parse(self):
        """Split the file into info block, header, and data."""
        info_start = self._find_line("Info-Block")
        info_end = self._find_line("Ende Info-Block")
        star_line = self._find_line("***")
        data_start = self._find_line("Start of Data")
        data_end = self._find_line("End of Data")

        self.info_title = self.lines[info_start].strip("# ").strip()
        self._parse_info_block(self.lines[info_start + 1:info_end])
        self._parse_header(self.lines[star_line + 1:data_start])
        data_lines = self.lines[data_start + 1:data_end]
        self.csv_text = "\n".join(data_lines) + "\n"
        self.data = self._parse_data(data_lines)

    def _find_line(self, marker):
        """Locate a marker line in the file."""
        for index, line in enumerate(self.lines):
            if marker in line:
                return index
        raise ValueError(f"Marker not found: {marker}")

    def _parse_info_block(self, block_lines):
        """Extract timestamp, metadata, configuration, and geometry."""
        section = "metadata"
        for line in block_lines:
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.startswith("<<<") and "Konfiguration" in stripped:
                section = "configuration"
                continue
            if re.match(r"^[A-Za-z]{3}-[A-Za-z]{3}-\d{2}-\d{4}", stripped):
                self.timestamp = stripped
                continue
            if ":" not in stripped:
                continue

            key, value = stripped.split(":", 1)
            key = re.sub(r"\s+", " ", key).strip()
            value = value.strip()
            if section == "configuration":
                if key.startswith(("Abstand", "Flaeche", "Geometrischer")):
                    self.geometry[key] = value
                else:
                    self.configuration[key] = value
            else:
                self.metadata[key] = value

    def _parse_header(self, header_lines):
        """Extract column names, units, and processing notes."""
        non_empty = [line.rstrip() for line in header_lines if line.strip()]
        if not non_empty:
            return
        self.column_names = [part.strip() for part in non_empty[0].split(",") if part.strip()]
        if len(non_empty) >= 2:
            units = re.findall(r"\[[^\]]+\]", non_empty[1])
            self.column_units = [""] + units
            while len(self.column_units) < len(self.column_names):
                self.column_units.append("")
        if len(non_empty) >= 3:
            notes = re.findall(r"(intpol(?:\s*\+\s*darkcorr)?|berechnet)", non_empty[2])
            self.column_notes = ["", ""] + notes
            while len(self.column_notes) < len(self.column_names):
                self.column_notes.append("")

    def _parse_data(self, data_lines):
        """Convert the data section into a measurement table."""
        rows = []
        for line in data_lines:
            if not line.strip():
                continue
            parts = [part.strip() for part in line.split(",")]
            if len(parts) != 5:
                continue
            rows.append(
                {
                    "index": int(parts[0]),
                    "wavelength_nm": float(parts[1]),
                    "mple_nA_nm_per_W": float(parts[2]),
                    "det_signal_nA": float(parts[3]),
                    "radiant_intensity_W_per_sr_nm": float(parts[4]),
                }
            )
        return pd.DataFrame(rows)

    @property
    def header(self):
        """Show column names with units and notes."""
        width = len(self.column_names)
        units = (self.column_units + [""] * width)[:width]
        notes = (self.column_notes + [""] * width)[:width]
        return pd.DataFrame({"name": self.column_names, "unit": units, "note": notes})

    @property
    def info_block(self):
        """Bundle the whole info block."""
        return {
            "title": self.info_title,
            "timestamp": self.timestamp,
            "metadata": self.metadata,
            "configuration": self.configuration,
            "geometry": self.geometry,
        }

    @property
    def csv_data(self):
        """Provide the parsed measurement table."""
        return self.data

    @property
    def wavelength_nm(self):
        """Provide the wavelength column."""
        return self.data["wavelength_nm"]

    @property
    def radiant_intensity(self):
        """Provide the radiant intensity column."""
        return self.data["radiant_intensity_W_per_sr_nm"]

    @property
    def det_signal(self):
        """Provide the detector signal column."""
        return self.data["det_signal_nA"]

    @property
    def mple(self):
        """Provide the MPLE column."""
        return self.data["mple_nA_nm_per_W"]

    @property
    def n_points(self):
        """Count the measurement points."""
        return len(self.data)

    @property
    def wavelength_range_nm(self):
        """Give the first and last wavelength of the scan."""
        values = self.data["wavelength_nm"]
        return float(values.iloc[0]), float(values.iloc[-1])

    def summary(self):
        """Summarize path, range, and key settings."""
        low, high = self.wavelength_range_nm
        return {
            "path": str(self.path),
            "timestamp": self.timestamp,
            "n_points": self.n_points,
            "wavelength_nm_min": low,
            "wavelength_nm_max": high,
            "dark_value_A": self.metadata.get("Dark-Value in A"),
            "aperture": self.configuration.get("AP-BLENDE"),
            "esp": self.configuration.get("ESP"),
        }


## Datei öffnen

Mit `from_path` liest und zerlegt die Klasse die Datei. `summary()` gibt einen kurzen Überblick.


In [ ]:
scan = SpectralEvalFile.from_path("../data/data-x1.txt")
print(scan.summary())


## Info-Block und Metadaten

`info_block` bündelt Titel, Zeitstempel, Metadaten, Konfiguration und Geometrie. Einzelne Teile erreichst du auch direkt.


In [ ]:
print(scan.info_block["title"])
print(scan.timestamp)
print(scan.metadata)
print(scan.configuration)
print(scan.geometry)


## Spaltenkopf

`header` zeigt Namen, Einheiten und Verarbeitungshinweise der fünf Spalten.


In [ ]:
print(scan.header)
print(scan.column_names)


## CSV-Daten

`csv_data` ist die Messreihe als DataFrame. Zusaetzlich gibt es Spalten-Properties und den Rohtext der Datenzeilen.


In [ ]:
print(scan.csv_data.head())
print("points", scan.n_points)
print("range_nm", scan.wavelength_range_nm)
print(scan.csv_text.splitlines()[0])


## Einzelne Messgrößen

Die Properties `wavelength_nm`, `mple`, `det_signal` und `radiant_intensity` greifen Spalten direkt ab.


In [ ]:
print(scan.wavelength_nm.head())
print(scan.radiant_intensity.max())
print(scan.det_signal.min())


## Kurzer Plot

Strahlstärke gegen Wellenlänge. Das zeigt, dass die geparsten Spalten zusammenpassen.


In [ ]:
import matplotlib.pyplot as plt

plt.plot(scan.wavelength_nm, scan.radiant_intensity)
plt.xlabel("wavelength_nm")
plt.ylabel("radiant_intensity_W_per_sr_nm")
plt.title("Spectral evaluation scan")
plt.show()


## Was die Klasse abdeckt

| Teil | Zugriff |
| --- | --- |
| gesamter Info-Block | `scan.info_block` |
| Zeitstempel | `scan.timestamp` |
| Metadaten | `scan.metadata` |
| Konfiguration | `scan.configuration` |
| Geometrie | `scan.geometry` |
| Spaltenkopf | `scan.header`, `scan.column_names` |
| Messdaten | `scan.csv_data` |
| Roh-CSV-Text | `scan.csv_text` |
| Einzelsäulen | `scan.wavelength_nm`, `scan.mple`, `scan.det_signal`, `scan.radiant_intensity` |
| Kurzüberblick | `scan.summary()` |
